In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
%config Completer.use_jedi = True

In [4]:
from langchain_openai import OpenAI

In [5]:
os.environ["OPENAI_API_KEY"]= os.getenv("chatgpt_key")

In [8]:
llm = OpenAI(temperature=0.6)
name = llm.invoke("I want to open an Italian restaurant. Please suggest a fency name for this.")
print(name)



"La Dolce Vita Trattoria" 


In [12]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(temperature=0.7)

prompt = PromptTemplate.from_template(
    "I want to open a {cuisine} restaurant. Suggest 5 fancy names."
)

chain = prompt | llm

response = chain.invoke({"cuisine": "Mexican"})

print(response.content)

1. Tequila Sunrise
2. Azul Agave
3. La Cucaracha
4. El Sol y la Luna
5. Pico de Oro


In [14]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

parser = JsonOutputParser()

prompt = PromptTemplate.from_template(
    """
    Suggest a restaurant idea for {cuisine}.
    Return in JSON format:
    {{
        "name": "",
        "menu": [],
        "location": ""
    }}
    """
)

chain = prompt | llm | parser

result = chain.invoke({"cuisine": "Italian"})

print(result)

{'name': 'Rustic Trattoria', 'menu': ['Homemade pasta dishes', 'Wood-fired pizzas', 'Antipasti platters', 'Tiramisu dessert'], 'location': 'Downtown area with outdoor patio seating'}


In [49]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
    give me top 2 rated Restaurants in the Toronto Downtown for the {cuisine} with it locations
    Return strictly in JSON format:
{{
    "restaurants": [
        {{
            "name": "",
            "location": ""
        }}
    ]
}}
    """)

chain = prompt | llm | parser

restaurant_names = chain.invoke({"cuisine":"Indian"})





In [50]:
restaurant_lists = [r['name'] for r in restaurant_names['restaurants']]
print(restaurant_lists)

['Khau Gully', 'Bindia Indian Bistro']


In [51]:
menu_prompt = PromptTemplate.from_template(
"""
For the restaurant "{restaurant_name}", give top 3 popular menu items.

Return in JSON:
{{
  "restaurant": "{restaurant_name}",
  "menu_items": ["", "", ""]
}}
"""
)

menu_chain = menu_prompt | llm | parser

output = [];

for restuarant in restaurant_lists:
    output.append(menu_chain.invoke({"restaurant_name": restuarant}))

print(output)

    

[{'restaurant': 'Khau Gully', 'menu_items': ['Samosa Chaat', 'Butter Chicken', 'Paneer Tikka']}, {'restaurant': 'Bindia Indian Bistro', 'menu_items': ['Butter Chicken', 'Samosa Chaat', 'Biryani']}]


In [52]:
restaurant_lists =  [x.strip() for x in restaurant_lists]
print(restaurant_lists)

['Khau Gully', 'Bindia Indian Bistro']


In [53]:
menu_prompt = PromptTemplate.from_template(
"""
You are given a list of restaurants.

{restaurants}

provide top 10 rated vegetarian menus for each restaurants with ratings

Return in JSON:
{{
  "results": [ 
  "restaurant": "",
  "menu_items": ["", "", ""]
  ]
}}
"""
)

menu_chain = menu_prompt | llm | parser

output = [];

output = menu_chain.invoke({"restaurants": restaurant_lists})

print(output)

    

{'results': [{'restaurant': 'Khau Gully', 'menu_items': [{'name': 'Paneer Tikka', 'rating': 4.5}, {'name': 'Vegetable Biryani', 'rating': 4.2}, {'name': 'Aloo Gobi', 'rating': 4.0}, {'name': 'Palak Paneer', 'rating': 4.3}, {'name': 'Chana Masala', 'rating': 4.1}, {'name': 'Vegetable Korma', 'rating': 4.4}, {'name': 'Mushroom Masala', 'rating': 4.2}, {'name': 'Dal Makhani', 'rating': 4.3}, {'name': 'Mixed Vegetable Curry', 'rating': 4.1}, {'name': 'Vegetable Jalfrezi', 'rating': 4.0}]}, {'restaurant': 'Bindia Indian Bistro', 'menu_items': [{'name': 'Saag Paneer', 'rating': 4.6}, {'name': 'Vegetable Samosa', 'rating': 4.4}, {'name': 'Malai Kofta', 'rating': 4.5}, {'name': 'Baingan Bharta', 'rating': 4.2}, {'name': 'Vegetable Pakora', 'rating': 4.3}, {'name': 'Paneer Butter Masala', 'rating': 4.7}, {'name': 'Vegetable Pulao', 'rating': 4.0}, {'name': 'Vegetable Jalfrezi', 'rating': 4.1}, {'name': 'Paneer Tikka Masala', 'rating': 4.3}, {'name': 'Navratan Korma', 'rating': 4.4}]}]}
